# FPINT GEMM Integration Tests

This notebook provides integration tests for FPINT GEMM implementations.
Tests compare all 5 implementations against reference and visualize results.

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path('.').absolute()))

import numpy as np
import matplotlib.pyplot as plt

from fpint_emul import *
from visualize import *
from test_utils import *

%matplotlib inline

## Error Functions

Available error functions:
- `relative_diff_percent`: Relative error in percentage (sensitive to small ref values)
- `absolute_diff`: Absolute difference
- `ulp_diff_fp16`: ULP (Units in Last Place) difference for FP16 (recommended for HW verification)

In [ ]:
# ERROR_FUNCS is now imported from test_utils
# Available: 'rel_percent', 'absolute', 'ulp_fp16'
print("Available error functions:", list(ERROR_FUNCS.keys()))

## Helper Functions & Data Classes

In [ ]:
# Helper functions and TestResult are now imported from test_utils:
# - to_float_matrix()
# - print_error_stats()
# - TestResult dataclass
# - run_single_test()
# - run_all_tests()
# - visualize_all_results()
# - plot_error_histogram()

print("TestResult fields:", [f.name for f in TestResult.__dataclass_fields__.values()])

## Test with ULP Error (ref_version=2, FP64 high-precision)

In [ ]:
# Run with ULP error (ref_version=2)
results_ulp_v2 = run_all_tests(
    M, K, N,
    ref_version=2,
    err_func=ulp_diff_fp16,
    err_name='ULP Error (FP16)',
    visualize=False
)

In [ ]:
visualize_all_results(results_ulp_v2)

## Error Distribution Analysis

In [ ]:
# Histogram of ULP errors for all implementations
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes_flat = axes.flatten()

for idx, (name, result) in enumerate(results_ulp_v2.items()):
    ax = axes_flat[idx]
    err_flat = result.error.flatten()
    
    ax.hist(err_flat, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(err_flat), color='r', linestyle='--', label=f'Mean={np.mean(err_flat):.2f}')
    ax.axvline(np.median(err_flat), color='g', linestyle='--', label=f'Median={np.median(err_flat):.2f}')
    ax.set_title(f'{name}')
    ax.set_xlabel('ULP Error')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

axes_flat[-1].axis('off')

fig.suptitle('ULP Error Distribution (ref_version=1)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# test